In [78]:
import pandas as pd
import joblib
from nba_api.stats.endpoints import leaguegamefinder,boxscoretraditionalv2,leaguedashplayerstats,leaguegamelog

In [79]:
model = joblib.load('nba_model_2024.pkl')
diff_features = joblib.load('model_features.pkl')
finder = leaguegamefinder.LeagueGameFinder(season_nullable="2024-25",league_id_nullable="00",season_type_nullable="Regular Season")
games_2425 = finder.get_data_frames()[0]

In [80]:
players_stats = leaguedashplayerstats.LeagueDashPlayerStats(season="2024-25").get_data_frames()[0]
stars = players_stats.sort_values('PTS', ascending=False).drop_duplicates('TEAM_ID')
stars_list = stars[['TEAM_ID', 'PLAYER_ID','PLAYER_NAME']]

player_logs = leaguegamelog.LeagueGameLog(
    season='2024-25', 
    player_or_team_abbreviation='P'
).get_data_frames()[0]

star_attendance = player_logs[player_logs['PLAYER_ID'].isin(stars_list['PLAYER_ID'])][['GAME_ID', 'PLAYER_ID']]
star_attendance['IS_ACTIVE'] = 1
print(stars_list.head())
print(star_attendance.head())

        TEAM_ID  PLAYER_ID              PLAYER_NAME
490  1610612760    1628983  Shai Gilgeous-Alexander
29   1610612750    1630162          Anthony Edwards
423  1610612743     203999             Nikola Jokić
180  1610612749     203507    Giannis Antetokounmpo
263  1610612738    1628369             Jayson Tatum
       GAME_ID  PLAYER_ID  IS_ACTIVE
2   0022400062       2544          1
22  0022400061    1628369          1
23  0022400062     203076          1
36  0022400061    1626157          1
41  0022400062    1630162          1


In [81]:
games_2425['POSS'] = 0.96*(games_2425['FGA']+games_2425['TOV']+0.4*games_2425['FTA']-games_2425['OREB'])
games_2425['eFG%'] = (games_2425['FGM']+.5*games_2425['FG3M'])/games_2425['FGA']
games_2425['TS%'] = games_2425['PTS']/(2*games_2425['FGA']+0.44*games_2425['FTA'])
games_2425['NET_RAT'] = 100*(games_2425['PTS']/(games_2425['POSS'])-(games_2425['PTS']-games_2425['PLUS_MINUS'])/(games_2425['POSS']))
games_2425.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2460 entries, 0 to 2459
Data columns (total 32 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   SEASON_ID          2460 non-null   object 
 1   TEAM_ID            2460 non-null   int64  
 2   TEAM_ABBREVIATION  2460 non-null   object 
 3   TEAM_NAME          2460 non-null   object 
 4   GAME_ID            2460 non-null   object 
 5   GAME_DATE          2460 non-null   object 
 6   MATCHUP            2460 non-null   object 
 7   WL                 2460 non-null   object 
 8   MIN                2460 non-null   int64  
 9   PTS                2460 non-null   int64  
 10  FGM                2460 non-null   int64  
 11  FGA                2460 non-null   int64  
 12  FG_PCT             2460 non-null   float64
 13  FG3M               2460 non-null   int64  
 14  FG3A               2460 non-null   int64  
 15  FG3_PCT            2460 non-null   float64
 16  FTM                2460 

In [82]:
games_2425['GAME_DATE'] = pd.to_datetime(games_2425['GAME_DATE'])
games_2425['CUM_WINS'] = games_2425.groupby('TEAM_ID')['WL'].transform(lambda x: (x == 'W').astype(int).cumsum().shift(1).fillna(0))
games_2425['GAMES_PLAYED'] = games_2425.groupby('TEAM_ID').cumcount().astype(int)
games_2425['SEASON_WIN_PCT'] = games_2425['CUM_WINS'] / games_2425['GAMES_PLAYED']
games_2425['SEASON_WIN_PCT'] = games_2425['SEASON_WIN_PCT'].fillna(0)
games_2425['DAYS_REST'] = games_2425.groupby('TEAM_ID')['GAME_DATE'].diff().dt.days
games_2425['DAYS_REST'] = games_2425['DAYS_REST'].clip(upper=4).fillna(4)
games_2425['IS_B2B'] = (games_2425['DAYS_REST'] <= 1).astype(int)
games_2425['IS_B2B'] = games_2425['IS_B2B'].fillna(0)

games_2425 = games_2425.dropna()
games_2425 = games_2425.sort_values(by=['TEAM_ID', 'GAME_DATE'])
cols_to_roll = ['AST', 'REB', 'TOV','TS%','NET_RAT','SEASON_WIN_PCT']
for col in cols_to_roll:
    games_2425['ROLL_' + col] = games_2425.groupby('TEAM_ID')[col].transform(lambda x: x.ewm(span=10).mean().shift(1))

from nba_api.stats.static import teams
team_abbr_to_id = {t['abbreviation']: t['id'] for t in teams.get_teams()}

games_2425['OPP_ABBR'] = games_2425['MATCHUP'].apply(lambda x: x.split(' ')[-1])
games_2425['OPP_ID'] = games_2425['OPP_ABBR'].map(team_abbr_to_id)

win_pct_map = games_2425.set_index(['TEAM_ID', 'GAME_DATE'])['SEASON_WIN_PCT'].to_dict()

games_2425['OPP_WIN_PCT_AT_GAME'] = games_2425.apply(
    lambda x: win_pct_map.get((x['OPP_ID'], x['GAME_DATE']), 0.5), axis=1
)

games_2425['ROLL_SOS'] = games_2425.groupby('TEAM_ID')['OPP_WIN_PCT_AT_GAME'].transform(
    lambda x: x.rolling(10).mean().shift(1).fillna(0.5)
)

games_2425['ADJ_NET_RAT'] = games_2425['ROLL_NET_RAT'] * games_2425['ROLL_SOS']


In [83]:
# 1. Clear the duplicate index issues by resetting
games_2425 = games_2425.sort_values(['TEAM_ID', 'GAME_DATE']).reset_index(drop=True)

# 2. Run the rolling logic using the unique index
# include_groups=False ensures future-proofing
games_2425['GAMES_IN_LAST_4_DAYS'] = games_2425.groupby('TEAM_ID').apply(
    lambda x: x.set_index('GAME_DATE')['WL'].rolling('4D').count().shift(1).fillna(0),
    include_groups=False
).values # Use .values to ignore index alignment entirely

games_2425['GAMES_IN_LAST_5_DAYS'] = games_2425.groupby('TEAM_ID').apply(
    lambda x: x.set_index('GAME_DATE')['WL'].rolling('5D').count().shift(1).fillna(0),
    include_groups=False
).values

# 3. Create Flags
games_2425['FATIGUE_3_IN_4'] = (games_2425['GAMES_IN_LAST_4_DAYS'] >= 3).astype(int)
games_2425['FATIGUE_4_IN_5'] = (games_2425['GAMES_IN_LAST_5_DAYS'] >= 4).astype(int)

In [84]:
stats_cols = ['GAME_ID','TEAM_NAME','TEAM_ID','WL','SEASON_WIN_PCT','DAYS_REST','GAMES_PLAYED','GAME_DATE','TS%','NET_RAT','IS_B2B','FATIGUE_3_IN_4','FATIGUE_4_IN_5','GAMES_IN_LAST_4_DAYS','GAMES_IN_LAST_5_DAYS','ADJ_NET_RAT'] + [col for col in games_2425.columns if 'ROLL_' in col]


home_games = games_2425[games_2425['MATCHUP'].str.contains('vs')][stats_cols].copy()
away_games = games_2425[games_2425['MATCHUP'].str.contains(' @ ')][stats_cols].copy()

home_games = home_games.add_prefix('HOME_').rename(columns={'HOME_GAME_ID': 'GAME_ID','HOME_GAME_DATE': 'GAME_DATE'})
away_games = away_games.add_prefix('AWAY_').rename(columns={'AWAY_GAME_ID': 'GAME_ID'})

model_df = pd.merge(home_games, away_games, on='GAME_ID')
model_df['HOME_WIN'] = (model_df['HOME_WL'] == 'W').astype(int)


model_df.columns
print(model_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1225 entries, 0 to 1224
Data columns (total 46 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   GAME_ID                    1225 non-null   object        
 1   HOME_TEAM_NAME             1225 non-null   object        
 2   HOME_TEAM_ID               1225 non-null   int64         
 3   HOME_WL                    1225 non-null   object        
 4   HOME_SEASON_WIN_PCT        1225 non-null   float64       
 5   HOME_DAYS_REST             1225 non-null   float64       
 6   HOME_GAMES_PLAYED          1225 non-null   int64         
 7   GAME_DATE                  1225 non-null   datetime64[ns]
 8   HOME_TS%                   1225 non-null   float64       
 9   HOME_NET_RAT               1225 non-null   float64       
 10  HOME_IS_B2B                1225 non-null   int64         
 11  HOME_FATIGUE_3_IN_4        1225 non-null   int64         
 12  HOME_F

In [85]:
star_map = dict(zip(stars_list['TEAM_ID'], stars_list['PLAYER_ID']))
model_df['HOME_STAR_ID'] = model_df['HOME_TEAM_ID'].map(star_map)
model_df['AWAY_STAR_ID'] = model_df['AWAY_TEAM_ID'].map(star_map)

# 1. Create the set for instant lookup (same as before)
active_pairs = set(zip(star_attendance['GAME_ID'], star_attendance['PLAYER_ID']))

In [86]:
for col in ['AST', 'REB', 'TOV','TS%','NET_RAT']:
    model_df[f'DIFF_{col}'] = model_df[f'HOME_ROLL_{col}'] - model_df[f'AWAY_ROLL_{col}']
model_df['DIFF_SEASON_WIN_PCT'] = model_df['HOME_SEASON_WIN_PCT'] - model_df['AWAY_SEASON_WIN_PCT']
model_df['DIFF_REST'] = model_df['HOME_DAYS_REST'] - model_df['AWAY_DAYS_REST']
model_df['DIFF_STAR_ACTIVE'] = [
    (1 if (game, h_star) in active_pairs else 0) - (1 if (game, a_star) in active_pairs else 0)
    for game, h_star, a_star in zip(model_df['GAME_ID'], model_df['HOME_STAR_ID'], model_df['AWAY_STAR_ID'])
]
model_df['DIFF_B2B'] = model_df['HOME_IS_B2B'] - model_df['AWAY_IS_B2B']
model_df['DIFF_GAMES_4D'] = model_df['HOME_GAMES_IN_LAST_4_DAYS'] - model_df['AWAY_GAMES_IN_LAST_4_DAYS']
model_df['DIFF_GAMES_5D'] = model_df['HOME_GAMES_IN_LAST_5_DAYS'] - model_df['AWAY_GAMES_IN_LAST_5_DAYS']
model_df['DIFF_3_IN_4'] = model_df['HOME_FATIGUE_3_IN_4'] - model_df['AWAY_FATIGUE_3_IN_4']
model_df['DIFF_4_IN_5'] = model_df['HOME_FATIGUE_4_IN_5'] - model_df['AWAY_FATIGUE_4_IN_5']
model_df['DIFF_ADJ_NET_RAT'] = model_df['HOME_ADJ_NET_RAT'] - model_df['AWAY_ADJ_NET_RAT']

model_df['HOME_FATIGUE_PENALTY'] = (
    1
    - 0.12 * model_df['HOME_IS_B2B']
    - 0.08 * model_df['HOME_FATIGUE_3_IN_4']
    - 0.05 * model_df['HOME_FATIGUE_4_IN_5']
    - 0.05 * model_df['HOME_GAMES_IN_LAST_5_DAYS'].clip(lower=2) / 3
)

model_df['AWAY_FATIGUE_PENALTY'] = (
    1
    - 0.12 * model_df['AWAY_IS_B2B']
    - 0.08 * model_df['AWAY_FATIGUE_3_IN_4']
    - 0.05 * model_df['AWAY_FATIGUE_4_IN_5']
    - 0.05 * model_df['AWAY_GAMES_IN_LAST_5_DAYS'].clip(lower=2) / 3
)

model_df['DIFF_FATIGUE_ADJ_NET_RAT'] = (
    model_df['HOME_ADJ_NET_RAT'] * model_df['HOME_FATIGUE_PENALTY']
    - model_df['AWAY_ADJ_NET_RAT'] * model_df['AWAY_FATIGUE_PENALTY']
)


diff_features = [col for col in model_df.columns if 'DIFF_' in col]

model_df['GAME_DATE'] = pd.to_datetime(model_df['GAME_DATE'])
model_df = model_df.sort_values('GAME_DATE').reset_index(drop=True)

X_2425 = model_df[diff_features]
y_actual = model_df['HOME_WIN']


In [87]:
preds_2425 = model.predict(X_2425)
probs_2425 = model.predict_proba(X_2425)[:, 1]

print(f"Current Season Prediction Accuracy: {(preds_2425 == y_actual).mean():.2%}")

Current Season Prediction Accuracy: 67.02%


In [88]:
# Create a results table for the 24-25 season
results_2425 = X_2425.copy()
results_2425['ACTUAL'] = y_actual
results_2425['PRED'] = preds_2425
results_2425['CONFIDENCE'] = probs_2425

# Check accuracy when the model is very sure (>70% probability)
high_conf = results_2425[results_2425['CONFIDENCE'] > 0.70]
print(f"Accuracy when >70% sure: {len(high_conf[high_conf['ACTUAL'] == high_conf['PRED']]) / len(high_conf):.2%}")

Accuracy when >70% sure: 76.88%


In [89]:
import plotly.figure_factory as ff
from sklearn.metrics import confusion_matrix

# Ensure we use the most recent predictions from your tuned model
# If the lengths don't match, re-run your X_test/y_test split
preds_2425 = model.predict(X_2425)
cm = confusion_matrix(y_actual, preds_2425)

# z is the matrix, x is the predicted labels, y is the actual labels
# We flip the matrix and y-axis to put 'Actual Win' at the bottom right
z = cm[::-1] 
x = ['Predicted Loss', 'Predicted Win']
y = ['Actual Loss', 'Actual Win'][::-1] 

fig = ff.create_annotated_heatmap(z, x=x, y=y, colorscale='Viridis')

fig.update_layout(
    title='<b>Confusion Matrix: Where is the model failing?</b>',
    xaxis_title='Model Prediction',
    yaxis_title='Actual Outcome',
    template='plotly_dark'
)
fig.show()

In [90]:
# Create a summary dataframe for analysis
audit_df = model_df[['GAME_DATE', 'HOME_TEAM_NAME', 'AWAY_TEAM_NAME', 'HOME_WIN']].copy()
audit_df['PROBABILITY'] = probs_2425 # This is the probability of a Home Win
audit_df['PRED'] = (audit_df['PROBABILITY'] > 0.5).astype(int)

# Filter for "Confident Mistakes"
# Case 1: Model was 80%+ sure Home would win, but they lost
# Case 2: Model was 80%+ sure Home would lose (prob < 20%), but they won
major_misses = audit_df[
    ((audit_df['PROBABILITY'] >= 0.80) & (audit_df['HOME_WIN'] == 0)) |
    ((audit_df['PROBABILITY'] <= 0.20) & (audit_df['HOME_WIN'] == 1))
].sort_values(by='PROBABILITY', ascending=False)

print(f"Number of 'Confident' Errors: {len(major_misses)}")
major_misses.head(5)

Number of 'Confident' Errors: 0


,GAME_DATE,HOME_TEAM_NAME,AWAY_TEAM_NAME,HOME_WIN,PROBABILITY,PRED


In [91]:
import plotly.express as px
import pandas as pd

# Use the best model found by your Grid Search
# If you didn't use Grid Search, change this to 'tuned_model'
best_model = model

# Get feature importance
importance = best_model.feature_importances_
feature_names = X_2425.columns # Use X_train to ensure names match the model

# Create a DataFrame for plotting
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importance}).sort_values(by='Importance', ascending=True)

# Plot with Plotly
fig = px.bar(fi_df, x='Importance', y='Feature', orientation='h',
             title='<b>Which Feature is the "MVP" of the Model?</b>',
             template='plotly_dark',
             color_continuous_scale='Viridis')
fig.show()

In [92]:
# 1. Identify the False Wins
audit_df = model_df.copy()
audit_df['PROBABILITY'] = probs_2425
audit_df['PRED'] = (audit_df['PROBABILITY'] > 0.5).astype(int)

# Filter for Predicted Win (1) but Actual Loss (0)
false_wins_df = audit_df[(audit_df['PRED'] == 1) & (audit_df['HOME_WIN'] == 0)].copy()

# 2. Select columns that reveal the "Why"
analysis_cols = [
    'GAME_DATE', 'HOME_TEAM_NAME', 'AWAY_TEAM_NAME', 
    'PROBABILITY', 'DIFF_ADJ_NET_RAT', 'DIFF_SEASON_WIN_PCT',
    'HOME_FATIGUE_3_IN_4', 'HOME_IS_B2B', 'HOME_DAYS_REST'
]

# Display the 10 games where the model was MOST confident but wrong
print("Deep Dive: Top 10 Overconfident False Wins")
false_wins_df[analysis_cols].sort_values(by='PROBABILITY', ascending=False).head(20)

Deep Dive: Top 10 Overconfident False Wins


,GAME_DATE,HOME_TEAM_NAME,AWAY_TEAM_NAME,PROBABILITY,DIFF_ADJ_NET_RAT,DIFF_SEASON_WIN_PCT,HOME_FATIGUE_3_IN_4,HOME_IS_B2B,HOME_DAYS_REST
1031,2025-03-19,Minnesota Timberwolves,New Orleans Pelicans,0.788530,12.193491,0.651515,1,1,-2.0
958,2025-03-10,Oklahoma City Thunder,Denver Nuggets,0.788530,8.067116,0.411765,0,1,-2.0
750,2025-02-06,Boston Celtics,Dallas Mavericks,0.788530,7.976394,0.433333,0,1,-2.0
644,2025-01-23,Oklahoma City Thunder,Dallas Mavericks,0.788530,8.067569,0.436700,0,1,-3.0
1158,2025-04-06,Oklahoma City Thunder,Los Angeles Lakers,0.788530,5.727451,0.500000,0,1,-2.0
388,2024-12-19,Detroit Pistons,Utah Jazz,0.788530,4.876351,0.414683,0,1,-2.0
435,2024-12-25,Boston Celtics,Philadelphia 76ers,0.784277,4.878886,0.509259,0,1,-2.0
809,2025-02-19,Los Angeles Lakers,Charlotte Hornets,0.778905,7.862232,0.448276,0,1,-1.0
443,2024-12-26,Milwaukee Bucks,Brooklyn Nets,0.774096,7.540142,0.334543,1,1,-2.0
707,2025-02-01,Minnesota Timberwolves,Washington Wizards,0.774096,17.135492,0.343137,1,1,-2.0


In [93]:
# Select only these 20 specific rows from your X_2425 matrix
top_misses_idx = [646, 952, 750, 958, 41, 710, 443, 494, 809, 1026] # add the rest of the indices
X_misses = X_2425.loc[top_misses_idx]

# Check the average values of the features for these misses
print("Average Stats for Major Misses:")
print(X_misses.mean())

Average Stats for Major Misses:
DIFF_AST                   -0.012580
DIFF_REB                    0.513644
DIFF_TOV                   -0.676481
DIFF_TS%                    0.020417
DIFF_NET_RAT                9.645557
DIFF_SEASON_WIN_PCT         0.258109
DIFF_REST                   0.100000
DIFF_STAR_ACTIVE            0.100000
DIFF_B2B                    0.000000
DIFF_GAMES_4D               0.300000
DIFF_GAMES_5D               0.300000
DIFF_3_IN_4                 0.200000
DIFF_4_IN_5                 0.000000
DIFF_ADJ_NET_RAT            4.805314
DIFF_FATIGUE_ADJ_NET_RAT    3.928747
dtype: float64


In [102]:
# Generate probabilities (raw)
model_df['HOME_WIN_PROB_RAW'] = model.predict_proba(X_2425)[:, 1]

# Binary prediction
model_df['HOME_WIN_PRED'] = (model_df['HOME_WIN_PROB_RAW'] >= 0.5).astype(int)

# Correct / incorrect
model_df['CORRECT'] = (model_df['HOME_WIN_PRED'] == model_df['HOME_WIN']).astype(int)
model_df[['HOME_WIN_PROB_RAW','HOME_WIN_PRED','CORRECT']].head()


,HOME_WIN_PROB_RAW,HOME_WIN_PRED,CORRECT
0,0.606952,1,1
1,0.666266,1,1
2,0.606952,1,0
3,0.714541,1,0
4,0.739516,1,0


In [111]:
high_conf_games = model_df[model_df['HOME_WIN_PROB_RAW'] >= 0.70]

print(high_conf_games[
    ['GAME_DATE', 'HOME_TEAM_NAME', 'AWAY_TEAM_NAME', 'HOME_WIN_PROB_RAW', 'HOME_WIN', 'CORRECT']
].sort_values('HOME_WIN_PROB_RAW', ascending=False))

     GAME_DATE         HOME_TEAM_NAME        AWAY_TEAM_NAME  \
138 2024-11-09    Cleveland Cavaliers         Brooklyn Nets   
241 2024-11-23        Milwaukee Bucks     Charlotte Hornets   
178 2024-11-15  Oklahoma City Thunder          Phoenix Suns   
166 2024-11-13  Oklahoma City Thunder  New Orleans Pelicans   
243 2024-11-24         Indiana Pacers    Washington Wizards   
..         ...                    ...                   ...   
656 2025-01-25          Chicago Bulls    Philadelphia 76ers   
102 2024-11-04          Chicago Bulls             Utah Jazz   
482 2025-01-01        New York Knicks             Utah Jazz   
221 2024-11-20      Memphis Grizzlies    Philadelphia 76ers   
136 2024-11-09      San Antonio Spurs             Utah Jazz   

     HOME_WIN_PROB_RAW  HOME_WIN  CORRECT  
138           0.788530         1        1  
241           0.788530         1        1  
178           0.788530         1        1  
166           0.788530         1        1  
243           0.788530 